# Amazon Fashion Collaborative Filtering Baseline (Academic / Non-Scalable)

This notebook is intentionally **educational**. It demonstrates traditional explicit-feedback collaborative filtering approaches and clearly documents why they are not suitable for your production-scale data.


## Problem Statement

Build an academic CF baseline on Amazon Fashion ratings using explicit techniques (SVD, KNNWithMeans), then highlight scalability limitations compared to the production ALS pipeline.

Key objective: show **how/why these methods fail at massive scale**, not just produce a score.


## Data

- `mini_train.csv` (~250K rows)
- `mini_test.csv` (~20K rows)
- Columns used: `user_id`, `parent_asin`, `rating`
- Columns ignored for this baseline: `timestamp`, `helpful_vote`

Even this micro-data is sparse enough to expose practical failure modes of memory-based CF.


In [ ]:
# Optional: run if scikit-surprise is not available
# %pip install scikit-surprise


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
PLOTTING_AVAILABLE = True
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    PLOTTING_AVAILABLE = False
    plt = None
    sns = None

pd.set_option("display.max_columns", 50)

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

SURPRISE_AVAILABLE = True
try:
    from surprise import Dataset, Reader, SVD, KNNWithMeans, accuracy
    from surprise.model_selection import train_test_split as surprise_train_test_split
except ImportError:
    SURPRISE_AVAILABLE = False

print(f"scikit-surprise available: {SURPRISE_AVAILABLE}")


In [ ]:
@dataclass
class Config:
    train_path: str = "mini_train.csv"
    test_path: str = "mini_test.csv"
    file_format: str = "csv"

    user_col: str = "user_id"
    item_col: str = "parent_asin"
    rating_col: str = "rating"

    rating_scale: Tuple[int, int] = (1, 5)
    positive_threshold: float = 4.0

    academic_sparse_threshold: int = 50
    knn_demo_threshold: int = 5

    ranking_k: int = 10
    sampled_negatives: int = 100
    ranking_max_users: Optional[int] = 2000
    random_state: int = 42


cfg = Config()
cfg


In [ ]:
def resolve_data_path(path: str) -> Path:
    raw = Path(path).expanduser()
    if raw.is_absolute() and raw.exists():
        return raw

    cwd = Path.cwd()
    candidates = [
        cwd / raw,
        cwd / raw.name,
        cwd / "research" / raw,
        cwd / "research" / raw.name,
    ]
    for parent in cwd.parents:
        candidates.extend([
            parent / raw,
            parent / raw.name,
            parent / "research" / raw,
            parent / "research" / raw.name,
        ])

    seen = set()
    deduped = []
    for c in candidates:
        key = str(c)
        if key not in seen:
            seen.add(key)
            deduped.append(c)

    for c in deduped:
        if c.exists():
            return c

    checked = "\n".join(f"- {p}" for p in deduped[:12])
    raise FileNotFoundError(
        f"Could not find file {path!r}.\n"
        f"Current working directory: {cwd}\n"
        f"Checked:\n{checked}"
    )


def read_interactions(path: str, file_format: str = "csv") -> pd.DataFrame:
    resolved = resolve_data_path(path)
    if file_format == "csv":
        return pd.read_csv(resolved)
    if file_format == "parquet":
        return pd.read_parquet(resolved)
    raise ValueError("file_format must be 'csv' or 'parquet'")


def prepare(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    cols = [cfg.user_col, cfg.item_col, cfg.rating_col]
    out = df[cols].copy()
    out[cfg.rating_col] = pd.to_numeric(out[cfg.rating_col], errors="coerce")
    out = out.dropna(subset=cols)
    out = out.groupby([cfg.user_col, cfg.item_col], as_index=False)[cfg.rating_col].max()
    return out


def matrix_density(df: pd.DataFrame, user_col: str, item_col: str) -> float:
    observed = len(df)
    possible = df[user_col].nunique() * df[item_col].nunique()
    return (observed / possible) if possible > 0 else 0.0


def gb_float(num_bytes: float) -> float:
    return num_bytes / (1024 ** 3)


def estimate_similarity_matrix_memory(n_entities: int, dtype_bytes: int = 4) -> float:
    # dense n x n matrix memory in GB
    return gb_float((n_entities ** 2) * dtype_bytes)


def build_user_item_sets(df: pd.DataFrame, user_col: str, item_col: str) -> Dict[str, Set[str]]:
    return df.groupby(user_col)[item_col].agg(set).to_dict()


def sampled_ranking_eval(
    model,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    cfg: Config,
) -> Dict[str, float]:
    rng = np.random.default_rng(cfg.random_state)

    user_col, item_col, rating_col = cfg.user_col, cfg.item_col, cfg.rating_col
    positives = test_df[test_df[rating_col] >= cfg.positive_threshold]
    user_to_pos = positives.groupby(user_col)[item_col].agg(set).to_dict()
    if not user_to_pos:
        return {f"HR@{cfg.ranking_k}": 0.0, f"NDCG@{cfg.ranking_k}": 0.0, f"MRR@{cfg.ranking_k}": 0.0, "eval_users": 0}

    users = sorted(user_to_pos.keys())
    if cfg.ranking_max_users is not None:
        users = users[: cfg.ranking_max_users]

    train_seen = build_user_item_sets(train_df, user_col, item_col)
    all_items = train_df[item_col].unique()

    hr, ndcg, mrr = [], [], []
    k = cfg.ranking_k

    for u in users:
        pos_items = user_to_pos[u]
        seen = train_seen.get(u, set())
        forbidden = seen.union(pos_items)

        negatives = []
        max_trials = cfg.sampled_negatives * 30
        trials = 0
        while len(negatives) < cfg.sampled_negatives and trials < max_trials:
            cand = all_items[int(rng.integers(0, len(all_items)))]
            if cand not in forbidden and cand not in negatives:
                negatives.append(cand)
            trials += 1

        candidates = list(pos_items) + negatives
        if len(candidates) < k:
            continue

        scored = []
        for iid in candidates:
            est = model.predict(u, iid).est
            scored.append((iid, est))

        scored.sort(key=lambda x: x[1], reverse=True)
        topk = [iid for iid, _ in scored[:k]]

        hit_ranks = [r for r, iid in enumerate(topk, start=1) if iid in pos_items]
        hr.append(1.0 if hit_ranks else 0.0)
        mrr.append((1.0 / hit_ranks[0]) if hit_ranks else 0.0)

        dcg = 0.0
        for rank, iid in enumerate(topk, start=1):
            if iid in pos_items:
                dcg += 1.0 / np.log2(rank + 1)

        ideal_hits = min(len(pos_items), k)
        idcg = float(np.sum([1.0 / np.log2(i + 2) for i in range(ideal_hits)])) if ideal_hits > 0 else 0.0
        ndcg.append((dcg / idcg) if idcg > 0 else 0.0)

    if not hr:
        return {f"HR@{k}": 0.0, f"NDCG@{k}": 0.0, f"MRR@{k}": 0.0, "eval_users": 0}

    return {
        f"HR@{k}": float(np.mean(hr)),
        f"NDCG@{k}": float(np.mean(ndcg)),
        f"MRR@{k}": float(np.mean(mrr)),
        "eval_users": int(len(hr)),
    }


In [ ]:
train_raw = read_interactions(cfg.train_path, cfg.file_format)
test_raw = read_interactions(cfg.test_path, cfg.file_format)

train_df = prepare(train_raw, cfg)
test_df = prepare(test_raw, cfg)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train users/items:", train_df[cfg.user_col].nunique(), train_df[cfg.item_col].nunique())
print("Test users/items :", test_df[cfg.user_col].nunique(), test_df[cfg.item_col].nunique())

display(train_df.head())


## EDA and Sparsity

Academic CF papers often begin with matrix density and long-tail analysis.

**Important limitation:** in real e-commerce, long-tail users are core business traffic. Filtering them out may improve benchmark numbers but hurts coverage, fairness, and revenue impact.


In [ ]:
train_density = matrix_density(train_df, cfg.user_col, cfg.item_col)
print(f"Train matrix density: {train_density:.8f}")

user_activity = train_df[cfg.user_col].value_counts()
print("Users with >=5 ratings :", int((user_activity >= 5).sum()))
print("Users with >=10 ratings:", int((user_activity >= 10).sum()))
print("Users with >=50 ratings:", int((user_activity >= 50).sum()))

if PLOTTING_AVAILABLE:
    plt.figure(figsize=(8, 4))
    sns.histplot(user_activity.clip(upper=20), bins=20)
    plt.title("User Rating Count Distribution (clipped at 20)")
    plt.xlabel("ratings per user")
    plt.ylabel("count")
    plt.show()
else:
    print("Plotting skipped: matplotlib/seaborn not installed.")

display(train_df[cfg.rating_col].value_counts().sort_index())


## Academic Sparsity Reduction (Users with >=50 Ratings)

This step is common in educational notebooks because KNN-style methods need denser data to look reasonable.

**Why this is unacceptable in production:**
- It discards the majority of real users (especially new/casual shoppers).
- It optimizes for benchmark convenience rather than business impact.
- It can make offline metrics look better while making live recommendations worse for most traffic.


In [ ]:
user_counts = train_df[cfg.user_col].value_counts()
dense_users_50 = user_counts[user_counts >= cfg.academic_sparse_threshold].index
train_dense_50 = train_df[train_df[cfg.user_col].isin(dense_users_50)].copy()

print("Rows after >=50 filter:", len(train_dense_50))
print("Users after >=50 filter:", train_dense_50[cfg.user_col].nunique())

if len(train_dense_50) == 0:
    print("No users meet >=50 on this micro dataset. This itself demonstrates long-tail sparsity.")
    print("For KNN demo only, falling back to users with >=5 ratings.")

dense_users_demo = user_counts[user_counts >= cfg.knn_demo_threshold].index
train_dense_demo = train_df[train_df[cfg.user_col].isin(dense_users_demo)].copy()

print("KNN demo rows (>=5):", len(train_dense_demo))
print("KNN demo users     :", train_dense_demo[cfg.user_col].nunique())
print("KNN demo items     :", train_dense_demo[cfg.item_col].nunique())
print(f"KNN demo density   : {matrix_density(train_dense_demo, cfg.user_col, cfg.item_col):.6f}")


## Why KNNWithMeans OOMs at Scale

KNNWithMeans relies on large similarity structures (user-user or item-item). Dense similarity memory grows as **O(N^2)**.

Even with `float32`, this quickly becomes tens/hundreds of GB. On your 11M-row production data, cardinality is larger than this micro sample, so memory pressure is worse.


In [ ]:
n_users = train_df[cfg.user_col].nunique()
n_items = train_df[cfg.item_col].nunique()

user_user_gb = estimate_similarity_matrix_memory(n_users, dtype_bytes=4)
item_item_gb = estimate_similarity_matrix_memory(n_items, dtype_bytes=4)

print(f"Unique users: {n_users:,}")
print(f"Unique items: {n_items:,}")
print(f"Dense user-user matrix (float32) ~ {user_user_gb:,.2f} GB")
print(f"Dense item-item matrix (float32) ~ {item_item_gb:,.2f} GB")
print("This excludes overhead, model state, and Python object costs.")


## Explicit Matrix Factorization with Surprise SVD

This is an **academic explicit-feedback baseline**. It is useful to document, but typically outperformed by modern scalable implicit ranking methods in production recommendation stacks.


In [ ]:
if not SURPRISE_AVAILABLE:
    raise ImportError("Install scikit-surprise first: pip install scikit-surprise")

reader = Reader(rating_scale=cfg.rating_scale)
train_data = Dataset.load_from_df(train_df[[cfg.user_col, cfg.item_col, cfg.rating_col]], reader)
trainset = train_data.build_full_trainset()

svd_model = SVD(n_factors=64, n_epochs=20, biased=True, random_state=cfg.random_state, verbose=False)
svd_model.fit(trainset)

train_users = set(train_df[cfg.user_col].unique())
train_items = set(train_df[cfg.item_col].unique())
test_warm = test_df[test_df[cfg.user_col].isin(train_users) & test_df[cfg.item_col].isin(train_items)].copy()

test_tuples = list(test_warm[[cfg.user_col, cfg.item_col, cfg.rating_col]].itertuples(index=False, name=None))
svd_preds = [svd_model.predict(uid, iid, r_ui=r) for uid, iid, r in test_tuples]

svd_rmse = accuracy.rmse(svd_preds, verbose=False)
svd_mae = accuracy.mae(svd_preds, verbose=False)

print("SVD warm-start test rows:", len(test_warm))
print(f"SVD RMSE: {svd_rmse:.4f}")
print(f"SVD MAE : {svd_mae:.4f}")


## Why RMSE Is Not Enough

RMSE measures rating-error magnitude, but e-commerce impact depends on **top-N ranking quality**.

Two models can have similar RMSE while producing very different top-10 recommendations.

So we also compute sampled ranking metrics: **HR@10**, **NDCG@10**, **MRR@10**.


In [ ]:
ranking_metrics = sampled_ranking_eval(svd_model, train_df, test_warm, cfg)
ranking_metrics


## KNNWithMeans Tiny Demo (Educational Only)

We run KNN on a tiny filtered subset only. Running this on full-scale production data is not realistic.

**Reason:** similarity-based methods scale poorly with user/item cardinality and often crash memory on large catalogs.


In [ ]:
if not SURPRISE_AVAILABLE:
    raise ImportError("Install scikit-surprise first: pip install scikit-surprise")

# Keep subset small for demonstration speed and memory.
active_users = train_dense_demo[cfg.user_col].value_counts().head(300).index
knn_demo_df = train_dense_demo[train_dense_demo[cfg.user_col].isin(active_users)].copy()

reader = Reader(rating_scale=cfg.rating_scale)
knn_data = Dataset.load_from_df(knn_demo_df[[cfg.user_col, cfg.item_col, cfg.rating_col]], reader)
knn_train, knn_valid = surprise_train_test_split(knn_data, test_size=0.2, random_state=cfg.random_state)

sim_options = {"name": "cosine", "user_based": False}
knn_model = KNNWithMeans(k=20, sim_options=sim_options, verbose=False)
knn_model.fit(knn_train)
knn_preds = knn_model.test(knn_valid)

knn_rmse = accuracy.rmse(knn_preds, verbose=False)
print("KNN demo rows:", len(knn_demo_df))
print(f"KNNWithMeans RMSE (tiny demo subset): {knn_rmse:.4f}")


## Key Takeaways

1. Academic sparsity filtering (e.g., users >=50 ratings) is convenient for demos but usually unacceptable for production coverage.
2. KNNWithMeans is educationally useful but memory-risky at realistic user/item cardinality because similarity storage grows quadratically.
3. RMSE/MAE are not sufficient for e-commerce recommendation decisions; ranking metrics (HR/NDCG/MRR) are closer to business relevance.
4. This notebook is a teaching baseline; the scalable production notebook should remain your operational default.
